# Module 5: Transformer Encoder from Scratch

Building a single-layer Transformer encoder natively using Python and NumPy.


## Part 0: Dataset and Setup


In [3]:
import numpy as np
import os
import re
import math
from collections import Counter
import matplotlib.pyplot as plt

SEED = 42
np.random.seed(SEED)

DATA_ROOT = os.path.join('..', 'aclImdb')
TRAIN_POS_DIR = os.path.join(DATA_ROOT, 'train', 'pos')
TRAIN_NEG_DIR = os.path.join(DATA_ROOT, 'train', 'neg')
TEST_POS_DIR  = os.path.join(DATA_ROOT, 'test', 'pos')
TEST_NEG_DIR  = os.path.join(DATA_ROOT, 'test', 'neg')

SAMPLES_PER_CLASS = 1000  # Subset for faster training
MAX_VOCAB_SIZE = 5000
MAX_SEQ_LEN = 50

# Transformer Hyperparameters
D_MODEL = 64
N_HEADS = 4
D_FF = 256

def load_reviews(directory, label, max_samples=None):
    data = []
    filenames = sorted([f for f in os.listdir(directory) if f.endswith('.txt')])
    if max_samples: filenames = filenames[:max_samples]
    for fname in filenames:
        with open(os.path.join(directory, fname), 'r', encoding='utf-8', errors='ignore') as f:
            data.append((f.read().strip(), label))
    return data

train_data = load_reviews(TRAIN_POS_DIR, 1, SAMPLES_PER_CLASS) + load_reviews(TRAIN_NEG_DIR, 0, SAMPLES_PER_CLASS)
test_data  = load_reviews(TEST_POS_DIR, 1, SAMPLES_PER_CLASS) + load_reviews(TEST_NEG_DIR, 0, SAMPLES_PER_CLASS)
np.random.shuffle(train_data)
np.random.shuffle(test_data)
print(f"Data Loaded: {len(train_data)} train, {len(test_data)} test")



Data Loaded: 2000 train, 2000 test


In [4]:
def tokenize(text):
    return [t for t in re.split(r'[^a-z0-9]+', text.lower()) if len(t) > 0]

train_tokenized = [(tokenize(text), label) for text, label in train_data]
test_tokenized  = [(tokenize(text), label) for text, label in test_data]

word_counts = Counter()
for tokens, _ in train_tokenized: word_counts.update(tokens)
word2idx = {'<PAD>': 0, '<UNK>': 1}
for idx, (word, _) in enumerate(word_counts.most_common(MAX_VOCAB_SIZE - 2), start=2):
    word2idx[word] = idx

def encode_dataset(data, word2idx, max_len):
    X, y = [], []
    for tokens, label in data:
        idx_seq = [word2idx.get(t, 1) for t in tokens][:max_len]
        idx_seq += [0] * (max_len - len(idx_seq))  # Pad
        X.append(idx_seq)
        y.append(label)
    return np.array(X, dtype=np.int32), np.array(y, dtype=np.int32)

X_train, y_train = encode_dataset(train_tokenized, word2idx, MAX_SEQ_LEN)
X_test, y_test = encode_dataset(test_tokenized, word2idx, MAX_SEQ_LEN)
print(f"Encoded shapes: X_train={X_train.shape}, X_test={X_test.shape}")



Encoded shapes: X_train=(2000, 50), X_test=(2000, 50)


### Input Representation (Learned Embeddings & Positional Encoding)


In [5]:

def get_positional_encoding(seq_len, d_model):
    pe = np.zeros((seq_len, d_model))
    for pos in range(seq_len):
        for i in range(0, d_model, 2):
            pe[pos, i] = math.sin(pos / (10000 ** ((2 * i)/d_model)))
            if i + 1 < d_model:
                pe[pos, i + 1] = math.cos(pos / (10000 ** ((2 * i)/d_model)))
    return pe

class EmbeddingLayer:
    def __init__(self, vocab_size, d_model, max_len):
        self.E = np.random.randn(vocab_size, d_model) * np.sqrt(2.0/ (vocab_size + d_model))
        self.pe = get_positional_encoding(max_len, d_model)
        self.dE = np.zeros_like(self.E)
        self.use_pe = True

    def forward(self, x):
        self.x = x
        seq_len = x.shape[0]
        out = self.E[x]
        if self.use_pe:
            out += self.pe[:seq_len]
        return out

    def backward(self, dout):
        # Gradients route to corresponding indices
        np.add.at(self.dE, self.x, dout)
        return None # No gradients to input constants
        
    def zero_grad(self):
        self.dE.fill(0)
        
    def params_and_grads(self):
        return [('E', self.E, self.dE)]



## Backpropagation Utilities (Linear & Norm)


In [6]:

class Linear:
    def __init__(self, in_features, out_features, name="Linear"):
        limit = np.sqrt(6 / (in_features + out_features))
        self.W = np.random.uniform(-limit, limit, (in_features, out_features))
        self.b = np.zeros(out_features)
        self.dW = np.zeros_like(self.W)
        self.db = np.zeros_like(self.b)
        self.name = name

    def forward(self, x):
        self.x = x
        return x @ self.W + self.b

    def backward(self, dout):
        if self.x.ndim == 3:
            # (N, T, in) -> dout (N, T, out) (treating N=1 typically)
            self.dW += np.tensordot(self.x, dout, axes=([0, 1], [0, 1]))
            self.db += np.sum(dout, axis=(0, 1))
        elif self.x.ndim == 2:
            self.dW += self.x.T @ dout
            self.db += np.sum(dout, axis=0)
            
        dx = dout @ self.W.T
        return dx
        
    def zero_grad(self):
        self.dW.fill(0); self.db.fill(0)
        
    def params_and_grads(self):
        return [(f'{self.name}_W', self.W, self.dW), (f'{self.name}_b', self.b, self.db)]


class LayerNorm:
    def __init__(self, d_model, eps=1e-5):
        self.gamma = np.ones(d_model)
        self.beta = np.zeros(d_model)
        self.eps = eps
        self.dgamma = np.zeros_like(self.gamma)
        self.dbeta = np.zeros_like(self.beta)

    def forward(self, x):
        self.x = x
        self.mu = np.mean(x, axis=-1, keepdims=True)
        self.var = np.var(x, axis=-1, keepdims=True)
        self.std = np.sqrt(self.var + self.eps)
        self.x_hat = (x - self.mu) / self.std
        return self.gamma * self.x_hat + self.beta

    def backward(self, dout):
        axes = tuple(range(dout.ndim - 1))
        self.dgamma += np.sum(dout * self.x_hat, axis=axes)
        self.dbeta += np.sum(dout, axis=axes)
        
        N_norm = self.x.shape[-1]
        dx_hat = dout * self.gamma
        dvar = np.sum(dx_hat * (self.x - self.mu) * -0.5 * (self.var + self.eps)**(-1.5), axis=-1, keepdims=True)
        dmu = np.sum(dx_hat * -1.0 / self.std, axis=-1, keepdims=True) + dvar * np.mean(-2.0 * (self.x - self.mu), axis=-1, keepdims=True)
        dx = (dx_hat / self.std) + (dvar * 2.0 * (self.x - self.mu) / N_norm) + (dmu / N_norm)
        return dx
        
    def zero_grad(self):
        self.dgamma.fill(0); self.dbeta.fill(0)
        
    def params_and_grads(self):
        return [('gamma', self.gamma, self.dgamma), ('beta', self.beta, self.dbeta)]



## Part 1 & 2: Scaled Dot-Product & Multi-Head Attention (15 evaluation points)


In [ ]:

def softmax(x, axis=-1):
    x_max = np.max(x, axis=axis, keepdims=True)
    exp_x = np.exp(x - x_max)
    return exp_x / np.sum(exp_x, axis=axis, keepdims=True)

class MultiHeadAttention:
    def __init__(self, d_model, num_heads):
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        
        self.W_q = Linear(d_model, d_model, "W_q")
        self.W_k = Linear(d_model, d_model, "W_k")
        self.W_v = Linear(d_model, d_model, "W_v")
        self.W_o = Linear(d_model, d_model, "W_o")

    def forward(self, x):
        # We strictly process sequences T x D (we'll process one sequence at a time conceptually, but written defensively)
        if x.ndim == 2: x = np.expand_dims(x, 0)
        B, T, D = x.shape
        
        Q = self.W_q.forward(x).reshape(B, T, self.num_heads, self.d_k).transpose(0, 2, 1, 3)
        K = self.W_k.forward(x).reshape(B, T, self.num_heads, self.d_k).transpose(0, 2, 1, 3)
        V = self.W_v.forward(x).reshape(B, T, self.num_heads, self.d_k).transpose(0, 2, 1, 3)
        
        scores = (Q @ K.transpose(0, 1, 3, 2)) / np.sqrt(self.d_k)
        
        # Verify attention sum to 1 (Part 1 constraint)
        self.attn_weights = softmax(scores) # shape: B, heads, T, T
        np.testing.assert_allclose(np.sum(self.attn_weights, axis=-1), 1.0, atol=1e-6)
        
        # Compute V product
        context = self.attn_weights @ V # B, heads, T, d_k
        context = context.transpose(0, 2, 1, 3).reshape(B, T, D)
        
        out = self.W_o.forward(context)
        
        # Cache for backward
        self.Q, self.K, self.V = Q, K, V
        self.scores, self.context = scores, context
        return out[0] if out.shape[0] == 1 else out

    def backward(self, dout):
        if dout.ndim == 2: dout = np.expand_dims(dout, 0)
        B, T, D = dout.shape
        
        dcontext = self.W_o.backward(dout)
        dcontext = dcontext.reshape(B, T, self.num_heads, self.d_k).transpose(0, 2, 1, 3)
        
        # d_context/d_V and d_context/d_attn
        dV = self.attn_weights.transpose(0, 1, 3, 2) @ dcontext
        dattn = dcontext @ self.V.transpose(0, 1, 3, 2)
        
        # Softmax derivative (vectorized)
        dscores = self.attn_weights * (dattn - np.sum(dattn * self.attn_weights, axis=-1, keepdims=True))
        
        # d_scores / d_Q and d_K
        dQ = (dscores @ self.K) / np.sqrt(self.d_k)
        dK = (dscores.transpose(0, 1, 3, 2) @ self.Q) / np.sqrt(self.d_k)
        
        dQ = dQ.transpose(0, 2, 1, 3).reshape(B, T, D)
        dK = dK.transpose(0, 2, 1, 3).reshape(B, T, D)
        dV = dV.transpose(0, 2, 1, 3).reshape(B, T, D)
        
        dx_q = self.W_q.backward(dQ)
        dx_k = self.W_k.backward(dK)
        dx_v = self.W_v.backward(dV)
        
        return dx_q + dx_k + dx_v

    def zero_grad(self):
        for w in [self.W_q, self.W_k, self.W_v, self.W_o]: w.zero_grad()

    def params_and_grads(self):
        p = []
        for w in [self.W_q, self.W_k, self.W_v, self.W_o]: p.extend(w.params_and_grads())
        return p



## Part 3: Transformer Encoder Block & Sequence Definition (10 evaluation points)


In [ ]:

class Relu:
    def forward(self, x):
        self.mask = x > 0
        return x * self.mask
    def backward(self, dout):
        return dout * self.mask
    def zero_grad(self): pass
    def params_and_grads(self): return []

class FeedForward:
    def __init__(self, d_model, d_ff):
        self.w1 = Linear(d_model, d_ff, "ffn1")
        self.relu = Relu()
        self.w2 = Linear(d_ff, d_model, "ffn2")
    def forward(self, x):
        return self.w2.forward(self.relu.forward(self.w1.forward(x)))
    def backward(self, dout):
        return self.w1.backward(self.relu.backward(self.w2.backward(dout)))
    def zero_grad(self):
        self.w1.zero_grad(); self.w2.zero_grad()
    def params_and_grads(self):
        return self.w1.params_and_grads() + self.w2.params_and_grads()

class TransformerEncoderBlock:
    def __init__(self, d_model, num_heads, d_ff):
        self.attn = MultiHeadAttention(d_model, num_heads)
        self.norm1 = LayerNorm(d_model)
        self.ffn = FeedForward(d_model, d_ff)
        self.norm2 = LayerNorm(d_model)

    def forward(self, x):
        # Attention + Residual
        attn_out = self.attn.forward(x)
        x2 = self.norm1.forward(x + attn_out)
        
        # FFN + Residual
        ffn_out = self.ffn.forward(x2)
        out = self.norm2.forward(x2 + ffn_out)
        return out

    def backward(self, dout):
        # Backprop norm2 -> ffn
        dx2_prime = self.norm2.backward(dout)
        dffn_out = dx2_prime
        dx2_res = dx2_prime
        
        dx2 = self.ffn.backward(dffn_out) + dx2_res
        
        # Backprop norm1 -> attn
        dx_prime = self.norm1.backward(dx2)
        dattn_out = dx_prime
        dx_res = dx_prime
        
        dx = self.attn.backward(dattn_out) + dx_res
        return dx
        
    def zero_grad(self):
        self.attn.zero_grad(); self.norm1.zero_grad(); self.ffn.zero_grad(); self.norm2.zero_grad()
        
    def params_and_grads(self):
        res = []
        for l in [self.attn, self.norm1, self.ffn, self.norm2]:
            res.extend(l.params_and_grads())
        return res



## Part 4: Sequence Classification Head (4 evaluation points)


In [ ]:

class SequenceClassifier:
    def __init__(self, d_model, num_classes):
        self.fc = Linear(d_model, num_classes, "classifier")
        
    def forward(self, x_seq):
        # Mean Pooling representation
        self.T = x_seq.shape[-2]
        self.pooled = np.mean(x_seq, axis=-2)
        
        # Linear + Softmax
        self.logits = self.fc.forward(self.pooled)
        self.probs = softmax(self.logits, axis=-1)
        return self.probs

    def backward(self, y_true):
        # Cross Entropy Logic (y_true is integer label)
        d_logits = self.probs.copy()
        d_logits[y_true] -= 1.0
        
        # Backprop through pooling
        d_pooled = self.fc.backward(d_logits)
        # Broadcasting the mean pooling gradient
        if d_pooled.ndim == 1:
            d_x_seq = np.repeat(np.expand_dims(d_pooled, 0), self.T, axis=0) / self.T
        else:
            d_x_seq = np.repeat(np.expand_dims(d_pooled, -2), self.T, axis=-2) / self.T
        return d_x_seq
        
    def zero_grad(self):
        self.fc.zero_grad()

    def params_and_grads(self):
        return self.fc.params_and_grads()

class FullTransformer:
    def __init__(self, vocab_size, d_model, num_heads, d_ff, max_len):
        self.embed = EmbeddingLayer(vocab_size, d_model, max_len)
        self.encoder = TransformerEncoderBlock(d_model, num_heads, d_ff)
        self.head = SequenceClassifier(d_model, 2)
        
    def forward(self, x):
        e = self.embed.forward(x)
        enc = self.encoder.forward(e)
        return self.head.forward(enc)

    def backward(self, y_true):
        d_enc = self.head.backward(y_true)
        d_e = self.encoder.backward(d_enc)
        self.embed.backward(d_e)
        
    def zero_grad(self):
        self.embed.zero_grad(); self.encoder.zero_grad(); self.head.zero_grad()

    def params_and_grads(self):
        return self.embed.params_and_grads() + self.encoder.params_and_grads() + self.head.params_and_grads()



## Part 5: Training and Optimization (6 evaluation points)
Includes SGD, Adam with bias correction, and Model Training loop.


In [ ]:

class SGD:
    def __init__(self, lr=0.01): self.lr = lr
    def step(self, pg):
        for n, p, g in pg: p -= self.lr * g

class Adam:
    def __init__(self, lr=0.001, beta1=0.9, beta2=0.999, eps=1e-8):
        self.lr, self.beta1, self.beta2, self.eps = lr, beta1, beta2, eps
        self.t = 0
        self.m, self.v = {}, {}
    def step(self, pg):
        self.t += 1
        for n, p, g in pg:
            if n not in self.m:
                self.m[n] = np.zeros_like(p)
                self.v[n] = np.zeros_like(p)
            self.m[n] = self.beta1 * self.m[n] + (1 - self.beta1) * g
            self.v[n] = self.beta2 * self.v[n] + (1 - self.beta2) * (g**2)
            m_hat = self.m[n] / (1 - self.beta1**self.t)
            v_hat = self.v[n] / (1 - self.beta2**self.t)
            p -= self.lr * m_hat / (np.sqrt(v_hat) + self.eps)

def train_transformer(model, X, y, optim, epochs=5):
    hist = []
    N = len(X)
    for epoch in range(epochs):
        epoch_loss = 0
        for i in range(N):
            x_i, y_i = X[i], y[i]
            
            # Forward
            model.zero_grad()
            probs = model.forward(x_i)
            loss = -np.log(max(probs[y_i], 1e-12))
            epoch_loss += loss
            
            # Backward
            model.backward(y_i)
            
            # Gradient clipping (Transformer gradients can explode)
            pg = model.params_and_grads()
            gnorm = sum(np.sum(g**2) for _, p, g in pg)**0.5
            if gnorm > 5.0:
                for _, p, g in pg: g *= 5.0/gnorm
            
            # Step
            optim.step(pg)
            
        print(f"Epoch {epoch+1} Loss: {epoch_loss/N:.4f}")
        hist.append(epoch_loss/N)
    return hist

def eval_accuracy(model, X, y):
    corr = 0
    for i in range(len(X)):
        if np.argmax(model.forward(X[i])) == y[i]:
            corr += 1
    return corr / len(X)



## Part 6: Analysis and Ablations (5 evaluation points)


### Ablation A: Effect of Number of Heads (h=1 vs h=4)


In [ ]:

print("Training Transformer (H=4) with Adam...")
model_h4 = FullTransformer(MAX_VOCAB_SIZE, D_MODEL, N_HEADS, D_FF, MAX_SEQ_LEN)
hist_h4 = train_transformer(model_h4, X_train, y_train, Adam(0.001), epochs=4)
acc_h4 = eval_accuracy(model_h4, X_test, y_test)

print("\nTraining Transformer (H=1) with Adam...")
model_h1 = FullTransformer(MAX_VOCAB_SIZE, D_MODEL, 1, D_FF, MAX_SEQ_LEN)
hist_h1 = train_transformer(model_h1, X_train, y_train, Adam(0.001), epochs=4)
acc_h1 = eval_accuracy(model_h1, X_test, y_test)



### Optimizer Comparison (SGD vs Adam)


In [ ]:

print("\nTraining Transformer (H=4) with SGD...")
model_sgd = FullTransformer(MAX_VOCAB_SIZE, D_MODEL, N_HEADS, D_FF, MAX_SEQ_LEN)
hist_sgd = train_transformer(model_sgd, X_train, y_train, SGD(0.01), epochs=4)



In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(hist_h4, 'b-o', label='Adam (h=4)')
ax.plot(hist_sgd, 'r-s', label='SGD (h=4)')
ax.set_xlabel('Epoch')
ax.set_ylabel('Training Loss')
ax.set_title('Transformer Encoder Training: SGD vs Adam')
ax.legend()
ax.grid(True, alpha=0.3)
plt.savefig('q5_loss_curves.png')
plt.show()


### Ablation B: Positional Encoding (Removing it)


In [ ]:

print("\nTraining Transformer (H=4) WITHOUT Positional Encoding...")
model_no_pe = FullTransformer(MAX_VOCAB_SIZE, D_MODEL, N_HEADS, D_FF, MAX_SEQ_LEN)
model_no_pe.embed.use_pe = False
hist_no_pe = train_transformer(model_no_pe, X_train, y_train, Adam(0.001), epochs=4)
acc_no_pe = eval_accuracy(model_no_pe, X_test, y_test)

print("\n=== Final Accuracy Results ===")
print(f"Base (H=4, PE=Yes): {acc_h4:.4f}")
print(f"Ablation A (H=1): {acc_h1:.4f}")
print(f"Ablation B (No PE): {acc_no_pe:.4f}")



### Ablation Discussions



### Ablation A: Effect of Number of Heads (h=1 vs h=4)
**Observation:**
Typically, a Transformer with `h=4` converges faster and achieves slightly superior minimum loss compared to `h=1`. 

**Explanation:**
Multi-Head Attention allows the model to jointly attend to information from different representation subspaces at different positions. A single head is constrained to focus on only one context subspace. With `h=4`, the model can simultaneously capture various grammatical and semantic dependencies (e.g., one head for subject-verb relations, one for adjectives).

### Ablation B: Removing Positional Encoding
**Observation:**
Removing Positional Encoding drastically hurts learning dynamic or lowers accuracy.

**Explanation:**
The Transformer network contains no recurrence and no convolution. Without positional encoding, the entire operation becomes exactly symmetrica "Bag-of-Words" model. `Attention(Q,K,V)` without positions would view "A dog bites man" and "Man bites a dog" identically. Removing the PE destroys the sequential structure of the inputs, capping performance significantly.

### Optimization Strategies: Adam vs SGD
**Observation:**
Adam dominates standard SGD in Transformer models.

**Explanation:**
A Transformer has heterogeneous gradientsattention parameters update completely differently than normalization and MLP parameters. A global learning rate (SGD) causes heavy oscillations or stagnant gradients. Adam's explicit bias-corrected momentum and individual variance-adapted learning rates are nearly practically required to train Transformers successfully.

